In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

In [3]:
df=pd.read_csv("AIML Dataset.csv")

In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df["isFraud"].value_counts()

In [ ]:
df.isnull().sum()


In [ ]:
df.value_counts("type").plot(kind="bar",title="transactiontype",color="red")
plt.xlabel("transaction type")
plt.ylabel("count")
plt.show()

In [ ]:
fraudrate=df.groupby("type")["isFraud"].mean().sort_values(ascending=False)
fraudrate.plot(kind="bar",color="red",title="fraud rate")
plt.xlabel("transaction type")
plt.show()



In [ ]:
df["amount"].describe().astype(int)

In [ ]:
sns.histplot(np.log1p(df["amount"]),bins=100,color="red",kde=True)
plt.xlabel("Log of Amount")
plt.ylabel("Frequency")
plt.title("Distribution of Transaction Amounts")
plt.show()

In [ ]:
sns.boxplot(data=df[df["amount"]< 50000], x="isFraud", y="amount", palette="Set2" )
plt.show()

In [ ]:
df.columns


In [ ]:
df["balancedifferoriginal"]=df["oldbalanceOrg"]-df["newbalanceOrig"]
df["balancedifferdesti"]=df["newbalanceDest"]-df["oldbalanceDest"]


In [ ]:
(df["balancedifferoriginal"] < 0).sum()

In [ ]:
frauds_per_step=df[df["isFraud"] == 1]["step"].value_counts().sort_index()
sns.lineplot(x=frauds_per_step.index,y=frauds_per_step.values,color="red")
plt.show()

In [ ]:
top_senders=df["nameOrig"].value_counts().head(5)
top_receievers=df["nameDest"].value_counts().head(5)
sns.barplot(x=top_senders.index,y=top_senders.values,color="red")
plt.show()

In [ ]:
fraud_users=df[df["isFraud"]==1]["nameOrig"].value_counts().head(5)
 

In [ ]:
fraud_users


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV


In [4]:
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [5]:
df_model=df.drop(["step","nameOrig","nameDest","isFlaggedFraud"],axis=1)

In [6]:
df_model.head()

,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


In [6]:
catogorical=['type']
numerical=["amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"] 


In [7]:
y=df_model["isFraud"]
X=df_model.drop("isFraud",axis=1)

In [8]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,stratify=y)

In [9]:
preprocessor=ColumnTransformer(
    transformers=[("nums",StandardScaler(),numerical),("cata",OneHotEncoder(drop="first"),catogorical)],remainder="drop"
)


In [15]:
model=XGBClassifier(tree_method="hist",scale_pos_weight=10)
param_grid = {
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [100, 200, 300]
}




In [16]:
best_pipeline=Pipeline(steps=[("preprocessor",preprocessor),("model",model)])
pipeline=RandomizedSearchCV(estimator=best_pipeline,param_distributions=param_grid, cv=3,n_jobs=-1,n_iter=10,random_state=42)
pipeline.fit(X_train,y_train)



,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': [0.01, 0.1, ...], 'model__max_depth': [3, 5, ...], 'model__n_estimators': [100, 200, ...]}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric e

In [17]:
best_pipeline=pipeline.best_estimator_

In [18]:
y_predict=best_pipeline.predict(X_test)
print(classification_report(y_test,y_predict))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906322
           1       0.90      0.78      0.84      2464

    accuracy                           1.00   1908786
   macro avg       0.95      0.89      0.92   1908786
weighted avg       1.00      1.00      1.00   1908786



In [19]:
confusion_matrix(y_test,y_predict)

array([[1906105,     217],
       [    541,    1923]])

In [20]:
pipeline.score(X_test,y_test)


0.9996028889566457

In [21]:
import joblib
joblib.dump(pipeline,"fraud_detection_model_XGBOOST.pkl")


['fraud_detection_model_XGBOOST.pkl']

In [ ]:
X.columns
